In [1]:
# Principal:    etl_service
# Role:         etl_service
# Catalog Role: etl_service_catalog_role
# Привилегии:   NAMESPACE_LIST (catalog)
#               bronze: TABLE_CREATE, TABLE_LIST, TABLE_READ_DATA,
#                       TABLE_WRITE_DATA, TABLE_FULL_METADATA
#               silver: TABLE_CREATE, TABLE_LIST, TABLE_READ_DATA,
#                       TABLE_WRITE_DATA, TABLE_FULL_METADATA
#
# Матрица доступов:
#   bronze: чтение + запись + создание + удаление (TABLE_FULL_METADATA включает DROP)
#   silver: чтение + запись + создание + удаление (TABLE_FULL_METADATA включает DROP)
#   gold:   НЕТ ДОСТУПА
#
# FORBIDDEN: SELECT gold, INSERT gold, CREATE gold

In [2]:
import os
from pyspark.sql import SparkSession

client_id = os.environ["ETL_SERVICE_CLIENT_ID"]
client_secret = os.environ["ETL_SERVICE_CLIENT_SECRET"]
credential = f"{client_id}:{client_secret}"

spark = SparkSession.builder \
    .appName("lakehouse-etl-service-rbac") \
    .config("spark.sql.catalog.lakehouse.credential", credential) \
    .getOrCreate()

spark

In [3]:
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.bronze").show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.silver").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|lakehouse    |
|spark_catalog|
+-------------+

+---------+---------------+-----------+
|namespace|tableName      |isTemporary|
+---------+---------------+-----------+
|bronze   |raw_products   |false      |
|bronze   |raw_customers  |false      |
|bronze   |raw_categories |false      |
|bronze   |raw_order_items|false      |
|bronze   |raw_orders     |false      |
+---------+---------------+-----------+

+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|silver   |customers  |false      |
|silver   |products   |false      |
|silver   |orders     |false      |
|silver   |order_items|false      |
+---------+-----------+-----------+



In [4]:
print("[ALLOWED] TABLE_LIST — lakehouse.bronze")
spark.sql("SHOW TABLES IN lakehouse.bronze").show(truncate=False)

[ALLOWED] TABLE_LIST — lakehouse.bronze
+---------+---------------+-----------+
|namespace|tableName      |isTemporary|
+---------+---------------+-----------+
|bronze   |raw_products   |false      |
|bronze   |raw_customers  |false      |
|bronze   |raw_categories |false      |
|bronze   |raw_order_items|false      |
|bronze   |raw_orders     |false      |
+---------+---------------+-----------+



In [5]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.bronze.raw_categories")
spark.sql("""
    SELECT
        id,
        name,
        description
    FROM lakehouse.bronze.raw_categories
    LIMIT 3
""").show(truncate=False)

[ALLOWED] TABLE_READ_DATA — lakehouse.bronze.raw_categories
+---+-----------+---------------------+
|id |name       |description          |
+---+-----------+---------------------+
|1  |Electronics|Электроника и гаджеты|
|2  |Clothing   |Одежда и аксессуары  |
|3  |Books      |Книги и журналы      |
+---+-----------+---------------------+



In [6]:
print("[ALLOWED] TABLE_FULL_METADATA — lakehouse.bronze.raw_orders")
spark.sql("DESCRIBE TABLE EXTENDED lakehouse.bronze.raw_orders").show(truncate=False)

[ALLOWED] TABLE_FULL_METADATA — lakehouse.bronze.raw_orders
+-----------------------------+------------------------------------------------------+-------+
|col_name                     |data_type                                             |comment|
+-----------------------------+------------------------------------------------------+-------+
|id                           |int                                                   |NULL   |
|customer_id                  |int                                                   |NULL   |
|status                       |string                                                |NULL   |
|total_amount                 |decimal(12,2)                                         |NULL   |
|order_date                   |string                                                |NULL   |
|                             |                                                      |       |
|# Metadata Columns           |                                                      

In [7]:
print("[ALLOWED] TABLE_WRITE_DATA — INSERT в lakehouse.bronze.raw_orders")
spark.sql("""
    INSERT INTO lakehouse.bronze.raw_orders
    VALUES (99999, 1, 'completed', CAST(100.00 AS DECIMAL(12,2)), '2026-01-01')
""")
print("INSERT ok")

spark.sql("""
    SELECT
        id,
        customer_id,
        status,
        total_amount,
        order_date
    FROM lakehouse.bronze.raw_orders
    WHERE id = 99999
""").show(truncate=False)

spark.sql("DELETE FROM lakehouse.bronze.raw_orders WHERE id = 99999")
print("DELETE ok")

[ALLOWED] TABLE_WRITE_DATA — INSERT в lakehouse.bronze.raw_orders
INSERT ok
+-----+-----------+---------+------------+----------+
|id   |customer_id|status   |total_amount|order_date|
+-----+-----------+---------+------------+----------+
|99999|1          |completed|100.00      |2026-01-01|
+-----+-----------+---------+------------+----------+

DELETE ok


In [8]:
print("[ALLOWED] TABLE_CREATE — lakehouse.bronze._test_bronze_probe")
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.bronze._test_bronze_probe")
except Exception:
    pass
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.bronze._test_bronze_probe (
        id   INT,
        name STRING
    ) USING iceberg
""")
print("CREATE ok")

[ALLOWED] TABLE_CREATE — lakehouse.bronze._test_bronze_probe
CREATE ok


In [9]:
spark.range(100) \
    .selectExpr("CAST(id AS INT) AS id", "CONCAT('test_', CAST(id AS STRING)) AS name") \
    .write.mode("append").saveAsTable("lakehouse.bronze._test_bronze_probe")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.bronze._test_bronze_probe
""").collect()[0]["cnt"]
print(f"Загружено строк: {cnt}")

spark.sql("""
    SELECT
        id,
        name
    FROM lakehouse.bronze._test_bronze_probe
    LIMIT 3
""").show(truncate=False)

Загружено строк: 100
+---+------+
|id |name  |
+---+------+
|0  |test_0|
|1  |test_1|
|2  |test_2|
+---+------+



In [10]:
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.bronze._test_bronze_probe")
    print("DROP ok")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"DROP пропущен (нет прав): {short}")

DROP ok


In [11]:
print("[ALLOWED] TABLE_LIST — lakehouse.silver")
spark.sql("SHOW TABLES IN lakehouse.silver").show(truncate=False)

[ALLOWED] TABLE_LIST — lakehouse.silver
+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|silver   |customers  |false      |
|silver   |products   |false      |
|silver   |orders     |false      |
|silver   |order_items|false      |
+---------+-----------+-----------+



In [12]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.silver.customers")
spark.sql("""
    SELECT
        id,
        name,
        email,
        city,
        created_at
    FROM lakehouse.silver.customers
    LIMIT 3
""").show(truncate=False)

[ALLOWED] TABLE_READ_DATA — lakehouse.silver.customers
+---+----------------------------+----------------------------+----------+----------+
|id |name                        |email                       |city      |created_at|
+---+----------------------------+----------------------------+----------+----------+
|1  |Козлова Ирина Ефимовна      |visheslavzinovev@example.org|д. Ребриха|2023-06-14|
|9  |Татьяна Кузьминична Соболева|nikola2019@example.net      |п. Кунгур |2023-10-11|
|14 |Фадеев Гостомысл Ааронович  |selivan_01@example.com      |п. Беслан |2024-01-29|
+---+----------------------------+----------------------------+----------+----------+



In [13]:
print("[ALLOWED] TABLE_FULL_METADATA — lakehouse.silver.orders")
spark.sql("DESCRIBE TABLE EXTENDED lakehouse.silver.orders").show(truncate=False)

[ALLOWED] TABLE_FULL_METADATA — lakehouse.silver.orders
+-----------------------------+--------------------------------------------------+-------+
|col_name                     |data_type                                         |comment|
+-----------------------------+--------------------------------------------------+-------+
|id                           |int                                               |NULL   |
|customer_id                  |int                                               |NULL   |
|status                       |string                                            |NULL   |
|total_amount                 |decimal(12,2)                                     |NULL   |
|order_date                   |date                                              |NULL   |
|                             |                                                  |       |
|# Metadata Columns           |                                                  |       |
|_spec_id                     |int

In [14]:
print("[ALLOWED] TABLE_WRITE_DATA — INSERT в lakehouse.silver.orders")
spark.sql("""
    INSERT INTO lakehouse.silver.orders
    VALUES (99998, 1, 'completed', CAST(99.99 AS DECIMAL(12,2)), CAST('2026-01-01' AS DATE))
""")
print("INSERT ok")

spark.sql("""
    SELECT
        id,
        customer_id,
        status,
        total_amount,
        order_date
    FROM lakehouse.silver.orders
    WHERE id = 99998
""").show(truncate=False)

spark.sql("DELETE FROM lakehouse.silver.orders WHERE id = 99998")
print("DELETE ok")

[ALLOWED] TABLE_WRITE_DATA — INSERT в lakehouse.silver.orders
INSERT ok
+-----+-----------+---------+------------+----------+
|id   |customer_id|status   |total_amount|order_date|
+-----+-----------+---------+------------+----------+
|99998|1          |completed|99.99       |2026-01-01|
+-----+-----------+---------+------------+----------+

DELETE ok


In [15]:
print("[ALLOWED] TABLE_CREATE — lakehouse.silver._test_silver_probe")
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.silver._test_silver_probe")
except Exception:
    pass
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.silver._test_silver_probe (
        id   INT,
        name STRING
    ) USING iceberg
""")
print("CREATE ok")

[ALLOWED] TABLE_CREATE — lakehouse.silver._test_silver_probe
CREATE ok


In [16]:
spark.range(100) \
    .selectExpr("CAST(id AS INT) AS id", "CONCAT('test_', CAST(id AS STRING)) AS name") \
    .write.mode("append").saveAsTable("lakehouse.silver._test_silver_probe")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.silver._test_silver_probe
""").collect()[0]["cnt"]
print(f"Загружено строк: {cnt}")

spark.sql("""
    SELECT
        id,
        name
    FROM lakehouse.silver._test_silver_probe
    LIMIT 3
""").show(truncate=False)

Загружено строк: 100
+---+------+
|id |name  |
+---+------+
|0  |test_0|
|1  |test_1|
|2  |test_2|
+---+------+



In [17]:
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.silver._test_silver_probe")
    print("DROP ok")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"DROP пропущен (нет прав): {short}")

DROP ok


In [18]:
print("[FORBIDDEN] SELECT из gold (gold закрыт для etl_service)")
try:
    spark.sql("""
        SELECT
            category_name,
            total_revenue
        FROM lakehouse.gold.mart_sales_by_category
        LIMIT 1
    """).show(truncate=False)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] INSERT в gold.mart_top_customers (gold закрыт)")
try:
    spark.sql("""
        INSERT INTO lakehouse.gold.mart_top_customers
        VALUES (99999, 'test', 1, CAST(1 AS BIGINT), CAST(100.00 AS DECIMAL(14,2)), 'Low')
    """)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] CREATE TABLE в gold (gold закрыт)")
try:
    spark.sql("""
        CREATE TABLE IF NOT EXISTS lakehouse.gold._test_probe (
            id   INT,
            name STRING
        ) USING iceberg
    """)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

[FORBIDDEN] SELECT из gold (gold закрыт для etl_service)
[ОЖИДАЕМО] Доступ запрещён: An error occurred while calling o43.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'etl_service' with activated PrincipalRoles '[etl_service]' and activated grants via '[etl_service, etl_service_catalog_role]' is not authorized for op LOAD_TABLE
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)

[FORBIDDEN] INSERT в gold.mart_top_customers (gold закрыт)
[ОЖИДАЕМО] Доступ запрещён: An error occurred while calling o43.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'etl_service' with activated PrincipalRoles '[etl_service]' and activated grants via '[etl_service, etl_service_catalog_role]' is not authorized for op LOAD_TABLE
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)

[FORBIDDEN] CREATE TABLE в gold (gold закрыт)
[ОЖИДАЕМО] Доступ запрещён: An error occ